# L4c: Shortest-Path Algorithms

__Which route from a source to a target has the smallest total cost?__ Shortest-path algorithms answer this question by accounting for edge weights; the route with the fewest edges need not be the least expensive.

In [L4a](../L4a/CHEME-5800-L4a-Lecture-GraphAndTreeRepresentations-Fall-2026.ipynb), we compared graph representations, and in [L4b](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb), we used traversal algorithms to explore reachable vertices. We now use weighted graphs to compare the costs of reaching those vertices and reconstruct routes that minimize the total weight.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Formulate the shortest-path problem:__ Define route cost in a weighted directed graph and distinguish a finite shortest-path distance from an unreachable target. Use edge relaxation to update distance estimates and record predecessors for route reconstruction.
> * __Explain Dijkstra's algorithm:__ Trace how a priority queue selects vertices and how edge relaxation updates their neighbors. Explain why nonnegative edge weights allow selected distances to be declared final, and describe the algorithm's computational cost.
> * __Explain Bellman–Ford's algorithm:__ Trace repeated relaxation passes and explain why at most $|\mathcal{V}|-1$ passes suffice when no reachable negative-weight cycle exists. Distinguish negative edges from negative-weight cycles and explain how the algorithm detects a reachable negative cycle.

In this lecture, we develop the edge-relaxation rule shared by both algorithms, then compare their update strategies and assumptions. We use small directed graphs to examine when the algorithms agree and what changes when negative weights are present.

Let's get started!

___


## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines the lecture folder path, and loads the course package and its dependencies.

Let's set up our code environment:


In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The course package provides the shortest-path algorithms in [`ShortestPathAlgorithms.jl`](../../../code/src/ShortestPathAlgorithms.jl). The [`PriorityQueue` type](https://juliacollections.github.io/DataStructures.jl/stable/priority-queue/) from [`DataStructures.jl`](https://juliacollections.github.io/DataStructures.jl/stable/) supports Dijkstra's algorithm, and the [`Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) supplies our checks. See the [Julia documentation](https://docs.julialang.org/en/v1/) for language details.

___


## Shortest-Path Problem and Relaxation

We focus on the __single-source shortest-path problem__: starting from one source vertex, find the minimum cost of reaching every other vertex. We first define a feasible route and its cost.

Let $\mathcal{G}=(\mathcal{V},\mathcal{E})$ be a directed graph with edge-weight function $w:\mathcal{E}\rightarrow\mathbb{R}$. A route from $s$ to $t$ is a walk that follows the edge directions:
$$
P=\langle v_0,v_1,\ldots,v_k\rangle,\qquad v_0=s,\quad v_k=t,
$$
where $(v_i,v_{i+1})\in\mathcal{E}$ for $i=0,\ldots,k-1$. Its total cost is:
$$
w(P):=\sum_{i=0}^{k-1}w(v_i,v_{i+1}).
$$

__When does a minimum route cost exist?__ For a reachable target, a finite minimum exists unless an $s$-to-$t$ route contains a cycle whose edge weights sum to a negative number. Repeating that cycle keeps lowering the route cost. Thus, no minimum-cost route exists.

> __Shortest-path distance:__
>
> Let $\mathcal{P}_{s,t}$ denote the set of feasible routes from $s$ to $t$. Provided that no negative-weight cycle lies on an $s$-to-$t$ route, the shortest-path distance is:
> $$
> d(s,t):=
> \begin{cases}
> \displaystyle\min_{P\in\mathcal{P}_{s,t}}w(P), & \mathcal{P}_{s,t}\neq\varnothing,\\[6pt]
> +\infty, & \mathcal{P}_{s,t}=\varnothing.
> \end{cases}
> $$
> The value $+\infty$ identifies an unreachable target. When the minimum is finite, any selected shortest path $P^*$ satisfies:
> $$
> P^*\in\arg\min_{P\in\mathcal{P}_{s,t}}w(P),\qquad w(P^*)=d(s,t).
> $$

A single-source algorithm computes these distances for every target. During the calculation, $\operatorname{dist}[v]$ stores the best distance found so far; at successful completion, it equals $d(s,v)$. The predecessor $\operatorname{prev}[v]$ records the preceding vertex on the selected route. Following these predecessors backward reconstructs the route, with $\operatorname{prev}[s]=\texttt{nothing}$ marking the source. When routes tie, the algorithm may select any one of them.

__How can we avoid checking every route?__ Shortest paths have [__optimal substructure__](https://ocw.mit.edu/courses/6-006-introduction-to-algorithms-fall-2011/7a2abc9bc568c743404e85e85cf6dc59_MIT6_006F11_lec15.pdf#page=6): if a shortest path from $s$ to $t$ passes through $u$, its prefix from $s$ to $u$ must also be a shortest path. Otherwise, replacing that prefix with a cheaper one would produce a cheaper path to $t$.

This property motivates __edge relaxation__, which tests whether extending a known route improves our current estimate for another vertex.

__What does it mean to relax an edge?__ We test whether reaching $v$ through $u$ gives a cheaper route than the best one found so far.

Initially, only the zero-edge route from $s$ to itself is known:
$$
\operatorname{dist}[v]=
\begin{cases}
0, & v=s,\\
+\infty, & v\ne s,
\end{cases}
\qquad
\operatorname{prev}[v]=\texttt{nothing}.
$$

If a finite route to $u$ has been discovered, appending the edge $(u,v)$ gives a candidate route with cost:
$$
\operatorname{alt}(u,v)=\operatorname{dist}[u]+w(u,v).
$$

> __Edge relaxation:__
>
> If $\operatorname{alt}(u,v)<\operatorname{dist}[v]$, update the distance and predecessor:
> $$
> \operatorname{dist}[v]\leftarrow\operatorname{alt}(u,v),
> \qquad
> \operatorname{prev}[v]\leftarrow u.
> $$
> Otherwise, retain both values. The strict comparison preserves the existing predecessor when routes tie.

__Why does this help?__ Every finite estimate is the cost of a discovered route. It therefore cannot be smaller than the minimum route cost:
$$
d(s,v)\leq\operatorname{dist}[v].
$$
Relaxation lowers an estimate when we discover a cheaper route. Immediately after relaxing $(u,v)$, the estimates satisfy:
$$
\operatorname{dist}[v]\leq\operatorname{dist}[u]+w(u,v).
$$

Dijkstra's algorithm selects the smallest tentative distance and declares it final; nonnegative edge weights guarantee that later routes cannot improve it. Bellman–Ford repeatedly relaxes every edge, allowing later passes to improve earlier estimates even when some weights are negative.

___


## Dijkstra's Algorithm
Dijkstra's algorithm solves the single-source problem by repeatedly selecting the unprocessed vertex with the smallest tentative distance. Its greedy step is valid when every edge weight is nonnegative.

__Initialize:__ Given $\mathcal{G}=(\mathcal{V},\mathcal{E})$, source $s\in\mathcal{V}$, and weights $w(u,v)\geq0$, set $\operatorname{dist}[v]\leftarrow+\infty$ and $\operatorname{prev}[v]\leftarrow\texttt{nothing}$ for every vertex $v\in\mathcal{V}$. Set $\operatorname{dist}[s]\leftarrow0$, create an empty processed set $\mathcal{S}$ and an empty min-priority queue $\mathcal{Q}$, and insert $s$ into $\mathcal{Q}$ with priority zero.

While $\mathcal{Q}\neq\varnothing$ __do__:

1. __Select a vertex__: Remove a vertex $u$ with the smallest priority from $\mathcal{Q}$.
2. __Mark it as processed__: If $u\in\mathcal{S}$, continue to the next iteration of the while loop. Otherwise, set $\mathcal{S}\leftarrow\mathcal{S}\cup\{u\}$.
3. __Relax its outgoing edges__: For each edge $(u,v)\in\mathcal{E}$:
    1. Compute the cost of reaching $v$ through $u$: $\operatorname{alt}\leftarrow\operatorname{dist}[u]+w(u,v)$.
    2. If $\operatorname{alt}<\operatorname{dist}[v]$, set $\operatorname{dist}[v]\leftarrow\operatorname{alt}$ and $\operatorname{prev}[v]\leftarrow u$. Insert $v$ into $\mathcal{Q}$ with priority $\operatorname{alt}$, or decrease its priority to $\operatorname{alt}$ if it is already in the queue.

__Return:__ When the queue is empty, return the distance map $\operatorname{dist}$ and predecessor map $\operatorname{prev}$.

__Why is this greedy choice valid?__ Suppose the next vertex has a tentative distance of __5__. Every other vertex waiting to be processed has a tentative distance of __at least 5__. Continuing through those vertices cannot produce a route costing less than 5, because each additional edge adds a nonnegative cost.

The proof's job is to establish that we have not overlooked a cheaper route to one of those waiting vertices. We prove this one vertex at a time, starting with the source: its distance is zero, and nonnegative edge weights prevent any route from having a lower cost.

> __Correctness of the greedy step:__
>
> Assume we have found the correct shortest-path distance for every previously selected vertex. Let $u$ be the next vertex selected, and suppose a cheaper path to $u$ exists.
>
> Along that path, let $y$ be the first vertex we have not yet selected and $x$ its predecessor. Because $x$ was already processed, relaxing $(x,y)$ gave $y$ an estimate no greater than the cost of the path prefix to $y$.
>
> The remaining edges have nonnegative weights, so that prefix cannot cost more than the full path to $u$. Thus, $y$ would have a smaller tentative distance than $u$, contradicting our choice of $u$. Therefore, the selected estimate is the shortest-path distance:
> $$
> \operatorname{dist}[u]=d(s,u).
> $$

A negative edge can invalidate this argument by lowering the cost later in a route. Our implementation therefore rejects graphs containing negative edges.

__What is the computational cost?__ With adjacency lists and a binary-heap priority queue, the running time is:
$$
\mathcal{O}\!\left((|\mathcal{V}|+|\mathcal{E}|)\log|\mathcal{V}|\right).
$$
Each reachable vertex is processed once, each outgoing edge is examined once, and queue insertions, priority updates, and minimum removals take logarithmic time.

___


## Bellman–Ford Algorithm
Bellman–Ford allows negative edge weights by repeatedly relaxing every edge. Each pass uses the current distance estimates to look for cheaper routes.

__Initialize:__ Given $\mathcal{G}=(\mathcal{V},\mathcal{E})$, source $s\in\mathcal{V}$, and weights $w(u,v)\in\mathbb{R}$, set $\operatorname{dist}[v]\leftarrow+\infty$ and $\operatorname{prev}[v]\leftarrow\texttt{nothing}$ for every vertex $v\in\mathcal{V}$. Then set $\operatorname{dist}[s]\leftarrow0$.

For each pass $k=1,2,\ldots,|\mathcal{V}|-1$ __do__:

1. __Reset the change flag__: Set $\texttt{changed}\leftarrow\texttt{false}$.
2. __Relax every edge__: For each edge $(u,v)\in\mathcal{E}$:
    1. If $\operatorname{dist}[u]=+\infty$, skip this edge and continue to the next edge.
    2. Compute the cost of reaching $v$ through $u$: $\operatorname{alt}\leftarrow\operatorname{dist}[u]+w(u,v)$.
    3. If $\operatorname{alt}<\operatorname{dist}[v]$, set $\operatorname{dist}[v]\leftarrow\operatorname{alt}$, $\operatorname{prev}[v]\leftarrow u$, and $\texttt{changed}\leftarrow\texttt{true}$.
3. __Check for convergence__: If $\texttt{changed}=\texttt{false}$, stop the relaxation passes because no distance improved during the complete pass.

__Check for a negative-weight cycle:__ After the relaxation passes, visit every edge $(u,v)\in\mathcal{E}$ once more:

1. If $\operatorname{dist}[u]=+\infty$, skip this edge and continue to the next edge.
2. If $\operatorname{dist}[u]+w(u,v)<\operatorname{dist}[v]$, report a reachable negative-weight cycle and stop without returning shortest-path distances.

__Return:__ If the cycle check finds no further improvement, return the distance map $\operatorname{dist}$ and predecessor map $\operatorname{prev}$.

__Why are $|\mathcal{V}|-1$ passes sufficient?__ Consider a route with three edges. By the end of the first pass, we have discovered a route to its first vertex after the source. By the end of the second pass, that information has reached the next vertex, and by the end of the third, the destination.

> __Number of relaxation passes:__
>
> After $k$ passes, the algorithm has considered the cost of every route containing at most $k$ edges.
>
> If no negative-weight cycle is reachable from the source, we can choose a shortest route that never repeats a vertex. Any repeated vertex creates a cycle, which we can remove without increasing the route's cost.
>
> Such a route visits at most $|\mathcal{V}|$ vertices and therefore contains at most $|\mathcal{V}|-1$ edges. Thus, $|\mathcal{V}|-1$ passes suffice to find every reachable vertex's shortest-path distance.

__How do we detect a negative-weight cycle?__ After $|\mathcal{V}|-1$ passes, check the edges once more. If an edge from a reachable vertex can still lower a distance estimate, a reachable negative-weight cycle exists. If we can reach the destination through a negative-weight cycle, going around the cycle again always makes the route cheaper. Thus, there is no shortest path to that destination.

__What is the computational cost?__ Each pass examines $|\mathcal{E}|$ edges, giving a worst-case running time of:
$$
\mathcal{O}(|\mathcal{V}|\,|\mathcal{E}|).
$$
The distance and predecessor maps require $\mathcal{O}(|\mathcal{V}|)$ storage. We can stop early when a complete pass makes no changes.

### Choosing a Shortest-Path Algorithm

When all edge weights are nonnegative, both algorithms return the same shortest-path distances. They may select different routes when several routes share the minimum cost.

| Feature | Dijkstra | Bellman–Ford |
|:--|:--|:--|
| Edge weights | Must be nonnegative | May be negative |
| Update strategy | Select the unprocessed vertex with the smallest distance estimate and relax its outgoing edges | Repeatedly relax every edge |
| Negative-cycle detection | Not supported | Detects negative-weight cycles reachable from the source |
| Worst-case time | $\mathcal{O}((\lvert\mathcal{V}\rvert+\lvert\mathcal{E}\rvert)\log\lvert\mathcal{V}\rvert)$ | $\mathcal{O}(\lvert\mathcal{V}\rvert\,\lvert\mathcal{E}\rvert)$ |
| Graph representation used here | Weighted adjacency list | Edge list |

Use Dijkstra when all edge weights are nonnegative. Use Bellman–Ford when negative edges may occur or when we need to detect a reachable negative-weight cycle.

___


## Worked Comparison
We first compare Dijkstra and Bellman–Ford on a directed graph with nonnegative weights, then examine what changes when negative weights are present.

The graph defined below has three routes from source vertex 1 to target vertex 4:

| Route | Total cost |
|:--|:--|
| $1\rightarrow2\rightarrow4$ | $4+1=5$ |
| $1\rightarrow3\rightarrow4$ | $1+5=6$ |
| $1\rightarrow3\rightarrow2\rightarrow4$ | $1+2+1=4$ |

Both algorithms should return the same shortest-path distances. Here, they should also reconstruct the same route to vertex 4 because the minimum-cost route is unique.


In [ ]:
# Build the nonnegative directed graph -
# Each tuple has the form (source vertex, target vertex, edge weight).
nonnegative_edges = weighted_edges([
    (1, 2, 4.0), # direct route from source 1 to vertex 2
    (1, 3, 1.0), # inexpensive first step on the optimal route
    (3, 2, 2.0), # reaches vertex 2 for total cost 1 + 2 = 3, improving the direct cost 4
    (2, 4, 1.0), # completes the optimal route to target 4 for total cost 4
    (3, 4, 5.0), # alternative from vertex 3 reaches target 4 for total cost 6
])
source_vertex, target_vertex = 1, 4 # compute from vertex 1 and reconstruct the route ending at vertex 4

# Compute the single-source solution with both valid algorithms -
dijkstra_result = dijkstra(nonnegative_edges, source_vertex)      # distances and predecessors from greedy finalization
bellman_result = bellman_ford(nonnegative_edges, source_vertex)   # distances and predecessors from repeated relaxation

# Reconstruct each route from its predecessor map -
shortest_path = reconstruct_path(dijkstra_result.previous, source_vertex, target_vertex) # walk backward, then reverse
bellman_path = reconstruct_path(bellman_result.previous, source_vertex, target_vertex)    # independent route check

# Display the route, target cost, and independent agreement checks -
(
    path = shortest_path,                                                        # ordered source-to-target vertex ids
    cost = path_cost(nonnegative_edges, shortest_path),                          # sum the three selected edge weights
    distances_agree = dijkstra_result.distances == bellman_result.distances,     # compare every source-to-vertex cost
    paths_agree = shortest_path == bellman_path,                                 # compare the reconstructed target routes
)

The output confirms that both algorithms find $1\rightarrow3\rightarrow2\rightarrow4$, with total cost 4. This route uses more edges than either alternative but has the lowest cost.

__What if every edge cost were 1?__ A route's total cost would equal its number of edges, so finding the shortest path would mean finding a route with the fewest edges. This is the problem we solve with breadth-first search in [L4b](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb). In this graph, the two-edge routes $1\rightarrow2\rightarrow4$ and $1\rightarrow3\rightarrow4$ would tie for the minimum cost of 2.

### Negative Edges and Reachable Negative Cycles
A negative edge can belong to a shortest path. In the first graph below, edge $(2,3)$ has weight $-2$, but the graph has no cycles. Bellman–Ford finds the shortest path $1\rightarrow2\rightarrow3\rightarrow4$, with cost $4-2+3=5$. Our Dijkstra implementation rejects this graph because it requires nonnegative edge weights.

The second graph contains the cycle $1\rightarrow2\rightarrow3\rightarrow1$, with total weight $1-2+0=-1$. Each trip around the cycle reduces the route cost by 1. We can therefore keep lowering the cost of reaching any vertex in this graph, so no shortest path exists. Bellman–Ford detects the negative-weight cycle and reports an error.


In [ ]:
# Build a graph with one negative edge but no negative-weight cycle -
# Bellman–Ford may use the negative edge because every source-to-target optimum remains finite.
negative_edge_graph = weighted_edges([
    (1, 2, 4.0),  # first step of the optimal route
    (1, 3, 5.0),  # direct alternative to vertex 3
    (2, 3, -2.0), # negative edge lowers the cost of reaching vertex 3 from 5 to 2
    (3, 4, 3.0),  # completes the optimal route to target 4 for total cost 5
])
negative_result = bellman_ford(negative_edge_graph, 1)              # no reachable negative cycle from source 1
negative_path = reconstruct_path(negative_result.previous, 1, 4)    # follow predecessors from target 4 back to source 1

# Build a reachable cycle with total weight 1 - 2 + 0 = -1 -
negative_cycle_graph = weighted_edges([
    (1, 2, 1.0),  # leave the source with cost 1
    (2, 3, -2.0), # reduce the running cost to -1
    (3, 1, 0.0),  # return to source 1 without offsetting the negative cost
])

# Capture expected errors so students can inspect them without stopping notebook execution -
dijkstra_rejection = try
    dijkstra(negative_edge_graph, 1) # must reject the -2 edge before running the greedy algorithm
    "unexpectedly accepted"         # sentinel reached only if the precondition check fails
catch error
    sprint(showerror, error)         # convert the expected ArgumentError into displayable text
end

cycle_rejection = try
    bellman_ford(negative_cycle_graph, 1) # an extra relaxation pass detects the reachable -1 cycle
    "unexpectedly accepted"              # sentinel reached only if cycle detection fails
catch error
    sprint(showerror, error)              # preserve the diagnostic while allowing later cells to run
end

# Display the valid Bellman–Ford route and both explicit rejections -
(
    path = negative_path,                                             # finite route 1 → 2 → 3 → 4
    cost = path_cost(negative_edge_graph, negative_path),             # 4 + (-2) + 3 = 5
    dijkstra_rejection = dijkstra_rejection,                          # negative-edge precondition message
    negative_cycle_rejection = cycle_rejection,                       # reachable-negative-cycle message
)

__What do we see?__ Bellman–Ford returns a path with cost 5 for the first graph. The captured error messages show Dijkstra rejecting the negative edge and Bellman–Ford detecting the negative-weight cycle in the second graph.


In [ ]:
# Check the numerical results and the public-interface contracts -
@testset "shortest-path contracts" begin
    # Nonnegative graph: route, cost, and both algorithms agree -
    @test shortest_path == [1, 3, 2, 4]                          # Dijkstra selects the three-edge minimum-cost route
    @test dijkstra_result.distances[4] == 4.0                    # target cost is 1 + 2 + 1
    @test dijkstra_result.distances == bellman_result.distances # both algorithms agree for every graph vertex
    @test shortest_path == bellman_path                          # the unique target route also agrees

    # Negative edge: Bellman–Ford still returns a finite optimum -
    @test negative_path == [1, 2, 3, 4]                          # the -2 edge belongs to the optimal route
    @test negative_result.distances[4] == 5.0                    # target cost is 4 - 2 + 3

    # Invalid inputs: each algorithm rejects the violated contract -
    @test_throws ArgumentError dijkstra(negative_edge_graph, 1)  # Dijkstra requires every edge weight to be nonnegative
    @test_throws ArgumentError bellman_ford(negative_cycle_graph, 1) # no finite optimum exists after a reachable -1 cycle
end

___

## Lab Exercises
In [L4d](../L4d/CHEME-5800-L4d-Lab-ProductionPlanningShortestPath-Fall-2026.ipynb), we compare two production routes: one uses fewer steps, while the other costs less. We calculate both route costs by hand, find the least-cost route with Dijkstra, and verify the result with Bellman–Ford.

___


## Summary
We developed two algorithms for finding minimum-cost routes and examined how edge weights determine which algorithm we can use.

> __Key Takeaways:__
>
> * **Route costs and relaxation:** We formulated the shortest-path problem by adding edge weights along a route. We used relaxation to improve distance estimates and recorded predecessors to reconstruct the selected route. When every edge costs 1, minimizing cost reduces to finding the fewest edges—the problem solved by breadth-first search.
>
> * **Dijkstra's algorithm:** We used a priority queue to select the unprocessed vertex with the smallest distance estimate. We showed why nonnegative edge weights make that estimate the correct shortest-path distance, and compared the result with Bellman–Ford on a graph with three competing routes.
>
> * **Bellman–Ford's algorithm:** We used repeated relaxation passes to find shortest paths when negative edges are present. We explained why $|\mathcal{V}|-1$ passes suffice when no negative-weight cycle is reachable from the source, and how an additional pass detects such a cycle. Our examples distinguished a negative edge that lowers a route's cost from a negative-weight cycle that prevents a shortest path from existing.

In L4d, we apply these ideas to a production network, where the route with the fewest steps may not be the least expensive.
